In [12]:
# Step 1: Install FFTW
!apt-get install libfftw3-dev
!apt-get install libfftw3-mpi-dev

# Step 2: Set paths for Dedalus installation
import os
import matplotlib.pyplot as plt
os.environ['MPI_INCLUDE_PATH'] = "/usr/lib/x86_64-linux-gnu/openmpi/include"
os.environ['MPI_LIBRARY_PATH'] = "/usr/lib/x86_64-linux-gnu"
os.environ['FFTW_INCLUDE_PATH'] = "/usr/include"
os.environ['FFTW_LIBRARY_PATH'] = "/usr/lib/x86_64-linux-gnu"

# Step 3: Install Dedalus using pip
!pip3 install --no-cache http://github.com/dedalusproject/dedalus/zipball/master/

#!pip3 install dedalus3

Reading package lists... Done
Building dependency tree       
Reading state information... Done
libfftw3-dev is already the newest version (3.3.8-2ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 15 not upgraded.
Reading package lists... Done
Building dependency tree       
Reading state information... Done
libfftw3-mpi-dev is already the newest version (3.3.8-2ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 15 not upgraded.
     \ 24.0 MB 19.7 MB/s 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for dedalus: filename=dedalus-3.0.0a0-cp310-cp310-linux_x86_64.whl size=2193670 sha256=729f7f12dea0cb21e113bd7ee4db67667b1a312ff435a22b621ad177ec64c22e
  Stored in directory: /tmp/pip-ephem-wheel-cache-pnlvkvgi/wheels/2c/16/80/8c5f40fc4de8dc37ba4cdd4e05d7639677a5f0659c3bbd79a7
Successfully built dedalus
  Attempting uninstall: dedalus
    Found existing installation: 

In [1]:
"""
Dedalus script simulating the viscous shallow water equations on a sphere. This
script demonstrates solving an initial value problem on the sphere. It can be
ran serially or in parallel, and uses the built-in analysis framework to save
data snapshots to HDF5 files. The `plot_sphere.py` script can be used to produce
plots from the saved data. The simulation should a few cpu-minutes to run.

The script implements the test case of a barotropically unstable mid-latitude
jet from Galewsky et al. 2004 (https://doi.org/10.3402/tellusa.v56i5.14436).
The initial height field balanced the imposed jet is solved with an LBVP.
A perturbation is then added and the solution is evolved as an IVP.

To run and plot using e.g. 4 processes:
    $ mpiexec -n 4 python3 shallow_water.py
    $ mpiexec -n 4 python3 plot_sphere.py snapshots/*.h5
"""

import numpy as np
import dedalus.public as d3
import logging
logger = logging.getLogger(__name__)


# Simulation units
meter = 1 / 6.37122e6
hour = 1
second = hour / 3600

# Parameters
Nphi = 256
Ntheta = 128
dealias = 3/2
R = 6.37122e6 * meter
Omega = 7.292e-5 / second
nu = 1e5 * meter**2 / second / 32**2 # Hyperdiffusion matched at ell=32
g = 9.80616 * meter / second**2
H = 1e4 * meter
timestep = 600 * second
stop_sim_time = 360 * hour
dtype = np.float64

# Bases
coords = d3.S2Coordinates('phi', 'theta')
dist = d3.Distributor(coords, dtype=dtype)
basis = d3.SphereBasis(coords, (Nphi, Ntheta), radius=R, dealias=dealias, dtype=dtype)

# Fields
u = dist.VectorField(coords, name='u', bases=basis)
h = dist.Field(name='h', bases=basis)

# Substitutions
zcross = lambda A: d3.MulCosine(d3.skew(A))

# Initial conditions: zonal jet
phi, theta = dist.local_grids(basis)
lat = np.pi / 2 - theta + 0*phi
umax = 80 * meter / second
lat0 = np.pi / 7
lat1 = np.pi / 2 - lat0
en = np.exp(-4 / (lat1 - lat0)**2)
jet = (lat0 <= lat) * (lat <= lat1)
u_jet = umax / en * np.exp(1 / (lat[jet] - lat0) / (lat[jet] - lat1))
u['g'][0][jet]  = u_jet

# Initial conditions: balanced height
c = dist.Field(name='c')
problem = d3.LBVP([h, c], namespace=locals())
problem.add_equation("g*lap(h) + c = - div(dot(u, grad(u)) + 2*Omega*zcross(u))")
problem.add_equation("ave(h) = 0")
solver = problem.build_solver()
solver.solve()

# Initial conditions: perturbation
lat2 = np.pi / 4
hpert = 120 * meter
alpha = 1 / 3
beta = 1 / 15
h['g'] += hpert * np.cos(lat) * np.exp(-(phi/alpha)**2) * np.exp(-((lat2-lat)/beta)**2)

# Problem
problem = d3.IVP([u, h], namespace=locals())
problem.add_equation("dt(u) + nu*lap(lap(u)) + g*grad(h) + 2*Omega*zcross(u) = - dot(u, grad(u))")
problem.add_equation("dt(h) + nu*lap(lap(h)) + H*div(u) = - div(h*u)")


# Solver
solver = problem.build_solver(d3.RK222)
solver.stop_sim_time = stop_sim_time

# Analysis
snapshots = solver.evaluator.add_file_handler('snapshots', sim_dt=1*hour, max_writes=10)
snapshots.add_task(h, name='height')
snapshots.add_task(-d3.div(d3.skew(u)), name='vorticity')

# Main loop
try:
    logger.info('Starting main loop')
    while solver.proceed:
        solver.step(timestep)
        if (solver.iteration-1) % 10 == 0:
            logger.info('Iteration=%i, Time=%e, dt=%e' %(solver.iteration, solver.sim_time, timestep))
except:
    logger.error('Exception raised, triggering end of main loop.')
    raise
finally:
    solver.log_stats()


2023-07-02 09:47:54,135 dedalus 0/1 WARNING :: Threading has not been disabled. This may massively degrade Dedalus performance.


2023-07-02 09:47:54,140 dedalus 0/1 WARNING :: We strongly suggest setting the "OMP_NUM_THREADS" environment variable to "1".


DEBUG:h5py._conv:Creating converter from 7 to 5
DEBUG:h5py._conv:Creating converter from 5 to 7
DEBUG:h5py._conv:Creating converter from 7 to 5
DEBUG:h5py._conv:Creating converter from 5 to 7
INFO:numexpr.utils:NumExpr defaulting to 8 threads.


2023-07-02 09:47:54,386 numexpr.utils 0/1 INFO :: NumExpr defaulting to 8 threads.


DEBUG:distributor:Mesh: []
DEBUG:transforms:Building FFTW FFT plan for (dtype, gshape, axis) = (<class 'numpy.float64'>, (2, 256, 128), 1)
DEBUG:problems:Adding equation 0
DEBUG:problems:  LHS: 19.947173947846725*Lap(h) + C(c)
DEBUG:problems:  RHS: -1*Div(u@Grad(u) + 0.525024*MulCos(Skew(u)))
DEBUG:problems:  condition: True
DEBUG:problems:  L: 19.947173947846725*Lap(h) + C(c)
DEBUG:problems:  F: -1*Div(u@Grad(u) + 0.525024*MulCos(Skew(u)))
DEBUG:problems:Adding equation 1
DEBUG:problems:  LHS: Average(h)
DEBUG:problems:  RHS: 0
DEBUG:problems:  condition: True
DEBUG:problems:  L: Average(h)
DEBUG:problems:  F: <Field 139986645193184>
DEBUG:solvers:Beginning LBVP instantiation
DEBUG:solvers:Finished LBVP instantiation
INFO:subsystems:Building subproblem matrices 1/127 (~1%) Elapsed: 0s, Remaining: 2s, Rate: 6.1e+01/s


...

INFO:solvers:CPU time (iter 10-end): 0.1012 cpu-hr


2023-07-02 09:54:18,856 solvers 0/1 INFO :: CPU time (iter 10-end): 0.1012 cpu-hr


INFO:solvers:Speed: 4.043e+05 mode-stages/cpu-sec


2023-07-02 09:54:18,858 solvers 0/1 INFO :: Speed: 4.043e+05 mode-stages/cpu-sec


KeyboardInterrupt: ignored

In [ ]:
zonal_unit = dist.VectorField(coords, name='zonal_unit', bases=basis)
zonal_unit['g'][0] = 1
zonal_unit['g'][1] = 0

np.shape((zonal_unit*u).evaluate()['g'])
np.shape((d3.dot(zonal_unit,u)*d3.skew(u)).evaluate()['g'])

(2, 384, 192)

In [ ]:
# Step 1: Install FFTW
!apt-get install libfftw3-dev
!apt-get install libfftw3-mpi-dev

# Step 2: Set paths for Dedalus installation
import os
os.environ['MPI_INCLUDE_PATH'] = "/usr/lib/x86_64-linux-gnu/openmpi/include"
os.environ['MPI_LIBRARY_PATH'] = "/usr/lib/x86_64-linux-gnu"
os.environ['FFTW_INCLUDE_PATH'] = "/usr/include"
os.environ['FFTW_LIBRARY_PATH'] = "/usr/lib/x86_64-linux-gnu"

# Step 3: Install Dedalus using pip
!pip3 install --no-cache http://github.com/dedalusproject/dedalus/zipball/d3/

Reading package lists... Done
Building dependency tree       
Reading state information... Done
The following packages were automatically installed and are no longer required:
  libnvidia-common-460 nsight-compute-2020.2.0
Use 'apt autoremove' to remove them.
The following additional packages will be installed:
  libfftw3-bin libfftw3-long3 libfftw3-quad3 libfftw3-single3
Suggested packages:
  libfftw3-doc
The following NEW packages will be installed:
  libfftw3-bin libfftw3-dev libfftw3-long3 libfftw3-quad3 libfftw3-single3
0 upgraded, 5 newly installed, 0 to remove and 42 not upgraded.
Need to get 3,766 kB of archives.
After this operation, 21.2 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu bionic/main amd64 libfftw3-long3 amd64 3.3.7-1 [308 kB]
Get:2 http://archive.ubuntu.com/ubuntu bionic/main amd64 libfftw3-quad3 amd64 3.3.7-1 [552 kB]
Get:3 http://archive.ubuntu.com/ubuntu bionic/main amd64 libfftw3-single3 amd64 3.3.7-1 [764 kB]
Get:4 http://ar

In [ ]:
#!mv /content/snapshots /content/drive/MyDrive/website-hugo/chaos_and_predictability/week3/RB2D/2D_shallow_water_output/snapshots
#!pip uninstall dedalus
!pip3 install --no-cache http://github.com/dedalusproject/dedalus/zipball/d3/

  ERROR: HTTP error 404 while getting http://github.com/dedalusproject/dedalus/zipball/d3/
ERROR: Could not install requirement http://github.com/dedalusproject/dedalus/zipball/d3/ because of HTTP error 404 Client Error: Not Found for url: https://codeload.github.com/DedalusProject/dedalus/legacy.zip/d3 for URL http://github.com/dedalusproject/dedalus/zipball/d3/


In [ ]:
# for plotting
!pip install dedalus

     |████████████████████████████████| 123 kB 4.1 MB/s 
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
    Preparing wheel metadata ... done
  Created wheel for dedalus: filename=dedalus-2.2006-cp37-cp37m-linux_x86_64.whl size=1289070 sha256=5ab90d3ead211c03f6130c8e908ce83f871e931dac23d31ece025506d0fa395a
  Stored in directory: /root/.cache/pip/wheels/5d/fb/3a/61d89cbb71c8f0a01732855dd91d8f087b278cc00781093b4f
Successfully built dedalus


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')



Mounted at /content/gdrive


In [ ]:
!ls /content/drive/MyDrive/website-hugo/chaos_and_predictability/week3/RB2D/2D_shallow_water_output/snapshots/snapshots_s*.h5

/content/drive/MyDrive/website-hugo/chaos_and_predictability/week3/RB2D/2D_shallow_water_output/snapshots/snapshots_s10.h5
/content/drive/MyDrive/website-hugo/chaos_and_predictability/week3/RB2D/2D_shallow_water_output/snapshots/snapshots_s11.h5
/content/drive/MyDrive/website-hugo/chaos_and_predictability/week3/RB2D/2D_shallow_water_output/snapshots/snapshots_s12.h5
/content/drive/MyDrive/website-hugo/chaos_and_predictability/week3/RB2D/2D_shallow_water_output/snapshots/snapshots_s13.h5
/content/drive/MyDrive/website-hugo/chaos_and_predictability/week3/RB2D/2D_shallow_water_output/snapshots/snapshots_s14.h5
/content/drive/MyDrive/website-hugo/chaos_and_predictability/week3/RB2D/2D_shallow_water_output/snapshots/snapshots_s15.h5
/content/drive/MyDrive/website-hugo/chaos_and_predictability/week3/RB2D/2D_shallow_water_output/snapshots/snapshots_s16.h5
/content/drive/MyDrive/website-hugo/chaos_and_predictability/week3/RB2D/2D_shallow_water_output/snapshots/snapshots_s17.h5
/content/drive/M

In [ ]:
!mv /content/drive/MyDrive/website-hugo/chaos_and_predictability/week3/RB2D/2D_shallow_water_output/snapshots /content/snapshots

In [ ]:
import pathlib
import dedalus.public as de
from dedalus.extras import flow_tools
from dedalus.tools import post
#Merging analysis files
post.merge_process_files("snapshots", cleanup=True)
set_paths = list(pathlib.Path("snapshots").glob("snapshots_s*.h5"))
post.merge_sets("snapshots/snapshots.h5", set_paths, cleanup=True)

2022-04-06 03:58:05,000 post 0/1 INFO :: Merging files from snapshots


ValueError: ignored

In [ ]:
def build_s2_coord_vertices(phi, theta):
    phi = phi.ravel()
    phi_vert = np.concatenate([phi, [2*np.pi]])
    phi_vert -= phi_vert[1] / 2
    theta = theta.ravel()
    theta_mid = (theta[:-1] + theta[1:]) / 2
    theta_vert = np.concatenate([[np.pi], theta_mid, [0]])
    return np.meshgrid(phi_vert, theta_vert, indexing='ij')

def main(filename, start, count, output):
    """Save plot of specified tasks for given range of analysis writes."""
    # Plot settings
    task = 'vorticity'
    cmap = plt.cm.RdBu_r
    dpi = 100
    figsize = (8, 8)
    savename_func = lambda write: 'write_{:06}.png'.format(write)
    # Create figure
    fig = plt.figure(figsize=figsize)
    ax = fig.add_axes([0, 0, 1, 1], projection='3d')
    # Plot writes
    with h5py.File(filename, mode='r') as file:
        dset = file['tasks'][task]
        phi = dset.dims[1][0][:].ravel()
        theta = dset.dims[2][0][:].ravel()
        phi_vert, theta_vert = build_s2_coord_vertices(phi, theta)
        x = np.sin(theta_vert) * np.cos(phi_vert)
        y = np.sin(theta_vert) * np.sin(phi_vert)
        z = np.cos(theta_vert)
        for index in range(start, start+count):
            data_slices = (index, slice(None), slice(None))
            data = dset[data_slices]
            clim = np.max(np.abs(data))
            norm = matplotlib.colors.Normalize(-clim, clim)
            fc = cmap(norm(data))
            #fc[:, theta.size//2, :] = [0,0,0,1]  # black equator
            if index == start:
                surf = ax.plot_surface(x, y, z, facecolors=fc, cstride=1, rstride=1, linewidth=0, antialiased=False, shade=False, zorder=5)
                ax.set_box_aspect((1,1,1))
                ax.set_xlim(-0.7, 0.7)
                ax.set_ylim(-0.7, 0.7)
                ax.set_zlim(-0.7, 0.7)
                ax.axis('off')
            else:
                surf.set_facecolors(fc.reshape(fc.size//4, 4))
            # Save figure
            savename = savename_func(file['scales/write_number'][index])
            savepath = output.joinpath(savename)
            fig.savefig(str(savepath), dpi=dpi)
    plt.close(fig)

In [ ]:
!rm list.txt
import os
for i in range(1,37):
    os.system('ls /content/drive/MyDrive/website-hugo/chaos_and_predictability/week3/RB2D/2D_shallow_water_output/snapshots/snapshots_s'+str(i)+'.h5 >> list.txt')

rm: cannot remove 'list.txt': No such file or directory


In [ ]:
!mkdir /content/drive/MyDrive/website-hugo/chaos_and_predictability/week3/RB2D/2D_shallow_water_output/figures

In [ ]:
with open('list.txt') as f:
     for line in f:
          newstr = line.strip()
          print(newstr)
          f   = h5py.File(newstr, 'r')

/content/drive/MyDrive/website-hugo/chaos_and_predictability/week3/RB2D/2D_shallow_water_output/snapshots/snapshots_s1.h5
/content/drive/MyDrive/website-hugo/chaos_and_predictability/week3/RB2D/2D_shallow_water_output/snapshots/snapshots_s2.h5
/content/drive/MyDrive/website-hugo/chaos_and_predictability/week3/RB2D/2D_shallow_water_output/snapshots/snapshots_s3.h5
/content/drive/MyDrive/website-hugo/chaos_and_predictability/week3/RB2D/2D_shallow_water_output/snapshots/snapshots_s4.h5
/content/drive/MyDrive/website-hugo/chaos_and_predictability/week3/RB2D/2D_shallow_water_output/snapshots/snapshots_s5.h5
/content/drive/MyDrive/website-hugo/chaos_and_predictability/week3/RB2D/2D_shallow_water_output/snapshots/snapshots_s6.h5
/content/drive/MyDrive/website-hugo/chaos_and_predictability/week3/RB2D/2D_shallow_water_output/snapshots/snapshots_s7.h5
/content/drive/MyDrive/website-hugo/chaos_and_predictability/week3/RB2D/2D_shallow_water_output/snapshots/snapshots_s8.h5
/content/drive/MyDrive/w

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
import matplotlib


def build_s2_coord_vertices(phi, theta):
    phi = phi.ravel()
    phi_vert = np.concatenate([phi, [2*np.pi]])
    phi_vert -= phi_vert[1] / 2
    theta = theta.ravel()
    theta_mid = (theta[:-1] + theta[1:]) / 2
    theta_vert = np.concatenate([[np.pi], theta_mid, [0]])
    return np.meshgrid(phi_vert, theta_vert, indexing='ij')

count=0
with open('list.txt') as f:
     for line in f:
          newstr = line.strip()
          print(newstr)
          f   = h5py.File(newstr, 'r')
          vor = f['tasks']['vorticity']
          dim = np.shape(vor)
          # Plot settings
          task = 'vorticity'
          cmap = plt.cm.RdBu_r
          dpi = 100
          figsize = (8, 8)
          fig = plt.figure(figsize=figsize)
          ax = fig.add_axes([0, 0, 1, 1], projection='3d')
          #plt.contourf(vor[0,:,:].T)
          phi = vor.dims[1][0][:].ravel()
          theta = vor.dims[2][0][:].ravel()
          phi_vert, theta_vert = build_s2_coord_vertices(phi, theta)
          x = np.sin(theta_vert) * np.cos(phi_vert)
          y = np.sin(theta_vert) * np.sin(phi_vert)
          z = np.cos(theta_vert)
          for index in range(1, dim[0]):
              data_slices = (index, slice(None), slice(None))
              data = vor[data_slices]
              clim = np.max(np.abs(data))
              norm = matplotlib.colors.Normalize(-clim, clim)
              fc = cmap(norm(data))
              #fc[:, theta.size//2, :] = [0,0,0,1]  # black equator
              if index == 1:
                  surf = ax.plot_surface(x, y, z, facecolors=fc, cstride=1, rstride=1, linewidth=0, antialiased=False, shade=False, zorder=5)
                  #ax.set_box_aspect((1,1,1))
                  ax.set_xlim(-0.7, 0.7)
                  ax.set_ylim(-0.7, 0.7)
                  ax.set_zlim(-0.7, 0.7)
                  ax.axis('off')
              else:
                  surf.set_facecolors(fc.reshape(fc.size//4, 4))
              count=count+1
              fig.savefig("/content/drive/MyDrive/website-hugo/chaos_and_predictability/week3/RB2D/2D_shallow_water_output/figures/"+str(count).zfill(3)+'.png', dpi=dpi)
          plt.close(fig)

In [ ]:
import glob
from PIL import Image

# filepaths
fp_in = "/content/drive/MyDrive/website-hugo/chaos_and_predictability/week3/RB2D/2D_shallow_water_output/figures/*.png"
fp_out = "/content/drive/MyDrive/website-hugo/chaos_and_predictability/week3/RB2D/2D_shallow_water_output/figures/image.gif"

# https://pillow.readthedocs.io/en/stable/handbook/image-file-formats.html#gif
imgs = (Image.open(f) for f in sorted(glob.glob(fp_in)))
img = next(imgs)  # extract first image from iterator
img.save(fp=fp_out, format='GIF', append_images=imgs,
         save_all=True, duration=10, loop=0)

In [ ]:
for count in range(1,316):
    os.system('mv /content/drive/MyDrive/website-hugo/chaos_and_predictability/week3/RB2D/2D_shallow_water_output/figures/'+str(count)+'.png /content/drive/MyDrive/website-hugo/chaos_and_predictability/week3/RB2D/2D_shallow_water_output/figures/'+str(count).zfill(3)+'.png')

In [ ]:
f   = h5py.File('/content/drive/MyDrive/website-hugo/chaos_and_predictability/week3/RB2D/2D_shallow_water_output/snapshots/snapshots_s2.h5', 'r')
vor = f['tasks']['vorticity']

In [ ]:
def eq_eval(eq_str):
    return [eval(expr) for expr in split_equation(eq_str)]

In [ ]:
eq_eval("u = 0")

NameError: ignored